# GPT od podstaw

Warsztaty w ramach Funduszu Zdolni (30 kwietnia - 2 maja 2026).

## Notebook 5: Mini-GPT

**Inspiracja:** [Let's build GPT - Andrej Karpathy (YouTube)](https://www.youtube.com/watch?v=kCc8FmEb1nY) oraz [nanoGPT](https://github.com/karpathy/nanoGPT). Architektura z [Attention Is All You Need (2017)](https://arxiv.org/abs/1706.03762), wariant decoder-only jak w GPT-2.

Składamy pełny mini-GPT: kilka głów uwagi równolegle, warstwa feed-forward, residual + LayerNorm, kilka bloków jeden na drugim.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from livelossplot import PlotLosses

torch.manual_seed(1337)

### 1. Hiperparametry i dane

In [ ]:
block_size = 64
batch_size = 32
n_embd = 96       # musi być podzielne przez n_head
n_head = 4
n_layer = 4

with open("data/pan-tadeusz.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for i, c in enumerate(chars)}

data = torch.tensor([char2id[c] for c in text], dtype=torch.long)
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
test_data = data[split_idx:]

def losuj_paczke(split: str = "train") -> tuple[torch.Tensor, torch.Tensor]:
    zrodlo = train_data if split == "train" else test_data
    ix = torch.randint(0, len(zrodlo) - block_size - 1, (batch_size,))
    x = torch.stack([zrodlo[i:i+block_size] for i in ix])
    y = torch.stack([zrodlo[i+1:i+block_size+1] for i in ix])
    return x, y

### 2. Głowa uwagi (taka sama jak w Notebooku 4)

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd: int, head_size: int, block_size: int):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        k = self.key(x); q = self.query(x); v = self.value(x)
        wagi = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wagi = wagi.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wagi = F.softmax(wagi, dim=-1)
        return wagi @ v

### 3. Multi-Head Attention

Kilka głów równolegle - każda może wyłapywać inne wzorce. Wyniki sklejamy i przepuszczamy przez warstwę liniową.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

### 4. Feed-Forward

Po wymianie informacji (uwaga) każdy token osobno "przemyśliwa" zebraną informację. Wewnętrzny wymiar 4× to konwencja z oryginalnego transformera.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

### 5. Blok transformera

Uwaga + feed-forward, każde w wersji `x = x + warstwa(LayerNorm(x))`. Połączenia rezydualne (`x + ...`) i LayerNorm są niezbędne, by trening głębszej sieci się nie rozjechał.

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = MultiHeadAttention(n_embd, n_head, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = FeedForward(n_embd)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

### 6. MiniGPT

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, n_embd: int, n_head: int, n_layer: int, block_size: int):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T))
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.lm_head(x)

    @torch.no_grad()
    def generuj(self, idx: torch.Tensor, max_nowych: int, temperatura: float = 1.0) -> torch.Tensor:
        for _ in range(max_nowych):
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)[:, -1, :] / temperatura
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

In [ ]:
model = MiniGPT(vocab_size, n_embd, n_head, n_layer, block_size)
print(f"Liczba parametrów: {sum(p.numel() for p in model.parameters()):,}")

### 7. Trening

In [ ]:
def oblicz_strate(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    logits = model(x)
    B, T, V = logits.shape
    return F.cross_entropy(logits.view(B*T, V), y.view(B*T))

@torch.no_grad()
def strata_test() -> float:
    model.eval()
    s = sum(oblicz_strate(*losuj_paczke("test")).item() for _ in range(20)) / 20
    model.train()
    return s

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
liveloss = PlotLosses()

for it in range(3000):
    xb, yb = losuj_paczke("train")
    loss = oblicz_strate(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if it % 100 == 0:
        liveloss.update({"loss": loss.item(), "val_loss": strata_test()})
        liveloss.send()

### 8. Generowanie

In [ ]:
def generuj_tekst(start_str: str, dlugosc: int = 500, temperatura: float = 1.0) -> str:
    idx = torch.tensor([[char2id[c] for c in start_str]], dtype=torch.long)
    out = model.generuj(idx, max_nowych=dlugosc, temperatura=temperatura)
    return "".join(id2char[i.item()] for i in out[0])

print(generuj_tekst("Litwo, ojczyzno moja", dlugosc=600, temperatura=0.9))